### Compute Metrics -- LPR & WPR

Computes Line Pass Rate (LPR) and Word Pass Rate (WPR) over every `outputs/{model}.csv`
produced by `Open_Source_Models.ipynb` / `Closed_Source_Models.ipynb`, using
`compute_metrics.py` -- a port of the upstream language-confusion repo's
`compute_metrics.py`, adapted to use GlotLID instead of fastText's `lid.176` (see that
module's docstring for why, including the AfroLID literature comparison that informed
the choice).

**LPR** = fraction of completions with every line correctly identified as the target
language. **WPR** = fraction of language-correct completions with no English dictionary
word leakage -- only computed for non-Latin-script languages in our set (Amharic,
Tigrinya), matching the original repo's rationale that WPR is unreliable for
Latin-script languages (loanwords/cognates cause false positives).

### Setup

Run once, from the `african-language-confusion/` folder, in a virtual environment (keeps
these dependencies out of your global Python install):

```bash
python -m venv venv
venv\Scripts\activate      # Windows
source venv/bin/activate    # macOS/Linux
pip install -r requirements.txt
```

Then pick `venv` as this notebook's kernel before running the cells below.

First run downloads two things automatically, cached afterward: GlotLID's language-ID
model (~1.6GB, via `huggingface_hub`) and an English word list used for WPR (~1MB, saved
as `words` in this folder).

In [ ]:
import csv
import glob
import itertools

import pandas as pd

import compute_metrics as cm

### Compute metrics for every output file

In [4]:
rows = []
for path in sorted(glob.glob("outputs/*.csv")):
    df = pd.read_csv(path)
    group_key = lambda o: (o["task"], o["model"])
    outputs = sorted(df.to_dict("records"), key=group_key)
    for (task, model), grouped in itertools.groupby(outputs, key=group_key):
        all_metrics = cm.compute_all_metrics(list(grouped))
        for (source, lang), metrics in all_metrics.items():
            rows.append({
                "task": task,
                "model": model,
                "source": source,
                "language": lang,
                "lpr": metrics.get("lpr"),
                "wpr": metrics.get("wpr"),
            })

results = pd.DataFrame(rows)
results

,task,model,source,language,lpr,wpr
0,crosslingual,qwen,okapi,igbo,1.000000,None
1,crosslingual,qwen,sharegpt,igbo,0.333333,None
2,crosslingual,qwen,okapi,all,1.000000,None
3,crosslingual,qwen,sharegpt,all,0.333333,None
4,crosslingual,qwen,all,igbo,0.666667,None
5,crosslingual,qwen,all,all,0.666667,None
6,monolingual,qwen,dolly,igbo,1.000000,None
7,monolingual,qwen,dolly,all,1.000000,None
8,monolingual,qwen,all,igbo,1.000000,None
9,monolingual,qwen,all,all,1.000000,None


### Save results

In [ ]:
results.to_csv("outputs/metrics_summary.csv", index=False)
print(f"Saved {len(results)} rows to outputs/metrics_summary.csv")

### CLI equivalent -- percentage table per file

Same logic as `compute_metrics.py`'s `if __name__ == "__main__":` block, for when you'd
rather run it here than as `python compute_metrics.py outputs/some_model.csv` from a
terminal. Loops over every `outputs/*.csv` (like the cell above) and prints a
tab-separated table with LPR/WPR as percentages, matching the CLI's formatting.

In [ ]:
print("task", "model", "source", "language", "lpr", "wpr", sep="\t")

for path in sorted(glob.glob("outputs/*.csv")):
    with open(path, encoding="utf-8") as csv_file:
        reader = csv.DictReader(csv_file)
        outputs = list(reader)

    group_key = lambda output: (output["task"], output["model"])
    outputs = sorted(outputs, key=group_key)
    for (task, model), outputs_ in itertools.groupby(outputs, key=group_key):
        all_metrics = cm.compute_all_metrics(outputs_)

        for (source, lang), metrics in all_metrics.items():
            lpr = f"{metrics['lpr']:.2%}"
            wpr = f"{metrics['wpr']:.2%}" if "wpr" in metrics else "N/A"
            print(task, model, source, lang, lpr, wpr, sep="\t")

In [ ]:
metrics = cm.compute_metrics(["Ndị mmadụ nwere ike ịrụ ọrụ ubi dị iche iche iji enwe mmasị, \n mana n'ụzọ dị iche iche gbasara obi ike, echiche, ma ọ bụ ihe mere eme ha. \n Ọ bụrụ na ị nọ n'ụzọ dị mfe ma ọ bụ na ị nọ n'ụzọ na-enyere aka, ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe. Mgbe ụbọchị dị mfe, ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe. \n This is in English and the language identifier is supposed to catch it"], "igbo")


print("Metrics: ", metrics)

Completion:  Ndị mmadụ nwere ike ịrụ ọrụ ubi dị iche iche iji enwe mmasị, 
 mana n'ụzọ dị iche iche gbasara obi ike, echiche, ma ọ bụ ihe mere eme ha. 
 Ọ bụrụ na ị nọ n'ụzọ dị mfe ma ọ bụ na ị nọ n'ụzọ na-enyere aka, ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe. Mgbe ụbọchị dị mfe, ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe. 
 This is in English and the language identifier is supposed to catch it
Normalized Completion:  Ndị mmadụ nwere ike ịrụ ọrụ ubi dị iche iche iji enwe mmasị 
 mana nụzọ dị iche iche gbasara obi ike echiche ma ọ bụ ihe mere eme ha 
 Ọ bụrụ na ị nọ nụzọ dị mfe ma ọ bụ na ị nọ nụzọ naenyere aka ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe Mgbe ụbọchị dị mfe ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe 
 This is in English and the language identifier is supposed to catch it
Lines:  ['Ndị mmadụ nwere ike ịrụ ọrụ ubi dị iche iche iji enwe mmasị ', ' mana nụzọ dị iche iche gbasara obi ike echiche ma ọ bụ ihe mere eme ha ', ' Ọ bụrụ na ị nọ nụzọ dị mfe ma ọ bụ na ị nọ nụzọ naenyere aka ọ nwere

### Same test, but with 3 completions -- lpr as a fraction, not a blanket 0/1

With only 1 completion above, `lpr` can only land on 0.0 or 1.0 (denominator of 1). Below,
`compute_metrics()` gets 3 completions: two fully-Igbo (using the same lines confirmed
`ibo_Latn` above) and one with the English line mixed in (confirmed `eng_Latn` above).
Expect `lpr = 2/3 ≈ 0.667` (2 of 3 completions had zero line errors) while `acc` averages
each completion's own per-line correctness (`1.0`, `0.75`, `1.0` → `≈0.917`).

In [4]:
three_completions = [
    # Fully Igbo -> should pass (0 line errors)
    "Ndị mmadụ nwere ike ịrụ ọrụ ubi dị iche iche iji enwe mmasị, \n mana n'ụzọ dị iche iche gbasara obi ike, echiche, ma ọ bụ ihe mere eme ha. \n Ọ bụrụ na ị nọ n'ụzọ dị mfe ma ọ bụ na ị nọ n'ụzọ na-enyere aka, ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe. Mgbe ụbọchị dị mfe, ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe",

    # Same 3 Igbo lines plus the English line -> should fail (1 line error out of 4)
    "Ndị mmadụ nwere ike ịrụ ọrụ ubi dị iche iche iji enwe mmasị, \n mana n'ụzọ dị iche iche gbasara obi ike, echiche, ma ọ bụ ihe mere eme ha. \n Ọ bụrụ na ị nọ n'ụzọ dị mfe ma ọ bụ na ị nọ n'ụzọ na-enyere aka, ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe. Mgbe ụbọchị dị mfe, ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe. \n This is in English and the language identifier is supposed to catch it",

    # Fully Igbo again (subset of the same confirmed-Igbo lines) -> should pass (0 line errors)
    "Ndị mmadụ nwere ike ịrụ ọrụ ubi dị iche iche iji enwe mmasị, \n Ọ bụrụ na ị nọ n'ụzọ dị mfe ma ọ bụ na ị nọ n'ụzọ na-enyere aka, ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe. Mgbe ụbọchị dị mfe, ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe",
]

metrics = cm.compute_metrics(three_completions, "igbo")
print("Metrics for 3 completions: ", metrics)

Completion:  Ndị mmadụ nwere ike ịrụ ọrụ ubi dị iche iche iji enwe mmasị, 
 mana n'ụzọ dị iche iche gbasara obi ike, echiche, ma ọ bụ ihe mere eme ha. 
 Ọ bụrụ na ị nọ n'ụzọ dị mfe ma ọ bụ na ị nọ n'ụzọ na-enyere aka, ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe. Mgbe ụbọchị dị mfe, ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe
Normalized Completion:  Ndị mmadụ nwere ike ịrụ ọrụ ubi dị iche iche iji enwe mmasị 
 mana nụzọ dị iche iche gbasara obi ike echiche ma ọ bụ ihe mere eme ha 
 Ọ bụrụ na ị nọ nụzọ dị mfe ma ọ bụ na ị nọ nụzọ naenyere aka ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe Mgbe ụbọchị dị mfe ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe
Lines:  ['Ndị mmadụ nwere ike ịrụ ọrụ ubi dị iche iche iji enwe mmasị ', ' mana nụzọ dị iche iche gbasara obi ike echiche ma ọ bụ ihe mere eme ha ', ' Ọ bụrụ na ị nọ nụzọ dị mfe ma ọ bụ na ị nọ nụzọ naenyere aka ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe Mgbe ụbọchị dị mfe ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe']
line_tokens:  [['Ndị', 'mmadụ', 'nwere', 'ike', 'ịrụ', 

## Word Pass Rate

In [5]:
metrics = cm.compute_metrics(["ስለ ቢትኮይን በተጠቀሰው ጽሑፍ ላይ በመመርኮዝ ስንት ሀገሮች ቢትኮይን አግደዋል?"], "amharic")

print("Metrics (Amharic): ", metrics)

Completion:  ስለ ቢትኮይን በተጠቀሰው ጽሑፍ ላይ በመመርኮዝ ስንት ሀገሮች ቢትኮይን አግደዋል?
Normalized Completion:  ስለ ቢትኮይን በተጠቀሰው ጽሑፍ ላይ በመመርኮዝ ስንት ሀገሮች ቢትኮይን አግደዋል
Lines:  ['ስለ ቢትኮይን በተጠቀሰው ጽሑፍ ላይ በመመርኮዝ ስንት ሀገሮች ቢትኮይን አግደዋል']
line_tokens:  [['ስለ', 'ቢትኮይን', 'በተጠቀሰው', 'ጽሑፍ', 'ላይ', 'በመመርኮዝ', 'ስንት', 'ሀገሮች', 'ቢትኮይን', 'አግደዋል']]

A Line  ስለ ቢትኮይን በተጠቀሰው ጽሑፍ ላይ በመመርኮዝ ስንት ሀገሮች ቢትኮይን አግደዋል
Lang ID per line -> amh_Ethi
Metrics (Amharic):  {'acc': 1.0, 'lpr': 1.0, 'wpr': 1.0}


### Same test, but with 3 completions -- wpr as a fraction, not a blanket 0/1

Same idea as the lpr demo above: with 1 completion, `wpr` can only be 0.0 or 1.0. Below,
`compute_metrics()` gets 3 completions, all reusing the line confirmed `amh_Ethi` above (so
all 3 should still pass langid -> `lpr = 1.0`). One of them has the English word `"internet"`
appended (lowercase, >3 chars, so it's a candidate match in the English word list) --
that's the only one expected to register a word leak.

Expect `wpr = 1 - 1/3 ≈ 0.667` (1 of the 3 language-correct completions leaked an English
word). Run it to confirm -- if the extra English word happens to flip that completion's
langid label away from `amh_Ethi`, it would instead count against `lpr`, not `wpr`, so
worth checking the printed "Lang ID per line" output too.

In [ ]:
three_completions_amharic = [
    # Confirmed amh_Ethi above, no English word -> should pass langid, no word leak
    "ስለ ቢትኮይን በተጠቀሰው ጽሑፍ ላይ በመመርኮዝ ስንት ሀገሮች ቢትኮይን አግደዋል?",

    # Same confirmed line + an embedded English word -> should still pass langid, but leak "internet"
    "ስለ ቢትኮይን በተጠቀሰው ጽሑፍ ላይ በመመርኮዝ ስንት ሀገሮች ቢትኮይን አግደዋል በኢንተርኔት internet?",

    # Confirmed amh_Ethi again, no English word -> should pass langid, no word leak
    "ጽሑፍ ላይ በመመርኮዝ ስንት ሀገሮች ቢትኮይን አግደዋል",
]

metrics = cm.compute_metrics(three_completions_amharic, "amharic")
print("Metrics for 3 completions (Amharic): ", metrics)

Completion:  ስለ ቢትኮይን በተጠቀሰው ጽሑፍ ላይ በመመርኮዝ ስንት ሀገሮች ቢትኮይን አግደዋል?
Normalized Completion:  ስለ ቢትኮይን በተጠቀሰው ጽሑፍ ላይ በመመርኮዝ ስንት ሀገሮች ቢትኮይን አግደዋል
Lines:  ['ስለ ቢትኮይን በተጠቀሰው ጽሑፍ ላይ በመመርኮዝ ስንት ሀገሮች ቢትኮይን አግደዋል']
line_tokens:  [['ስለ', 'ቢትኮይን', 'በተጠቀሰው', 'ጽሑፍ', 'ላይ', 'በመመርኮዝ', 'ስንት', 'ሀገሮች', 'ቢትኮይን', 'አግደዋል']]

A Line  ስለ ቢትኮይን በተጠቀሰው ጽሑፍ ላይ በመመርኮዝ ስንት ሀገሮች ቢትኮይን አግደዋል
Lang ID per line -> amh_Ethi
Completion:  ስለ ቢትኮይን በተጠቀሰው ጽሑፍ ላይ በመመርኮዝ ስንት ሀገሮች ቢትኮይን አግደዋል በኢንተርኔት internet?
Normalized Completion:  ስለ ቢትኮይን በተጠቀሰው ጽሑፍ ላይ በመመርኮዝ ስንት ሀገሮች ቢትኮይን አግደዋል በኢንተርኔት internet
Lines:  ['ስለ ቢትኮይን በተጠቀሰው ጽሑፍ ላይ በመመርኮዝ ስንት ሀገሮች ቢትኮይን አግደዋል በኢንተርኔት internet']
line_tokens:  [['ስለ', 'ቢትኮይን', 'በተጠቀሰው', 'ጽሑፍ', 'ላይ', 'በመመርኮዝ', 'ስንት', 'ሀገሮች', 'ቢትኮይን', 'አግደዋል', 'በኢንተርኔት', 'internet']]

A Line  ስለ ቢትኮይን በተጠቀሰው ጽሑፍ ላይ በመመርኮዝ ስንት ሀገሮች ቢትኮይን አግደዋል በኢንተርኔት internet
Lang ID per line -> amh_Ethi
Completion:  English word is not supposed to pass
Normalized Completion:  English word is not supposed to pass
Lines:  ['English 

## compute_all_metrics() example

`compute_all_metrics()` takes a flat list of `{"source", "language", "completion"}` dicts
-- e.g. every row from one model's `outputs/{model}.csv` -- and groups/scores them all at
once, instead of calling `compute_metrics()` yourself per (source, language) pair.

Reusing the completions from the cells above (2 Igbo + 1 Igbo from a different source, and
2 Amharic), the returned dict has one entry per `(source, language)` pair plus the
aggregates described in its docstring:

- `("okapi", "igbo")` / `("okapi", "amharic")` / `("dolly", "igbo")` -- scores for that
  exact source+language group
- `("okapi", "all")` / `("dolly", "all")` -- averaged over all languages within that source
- `("all", "igbo")` / `("all", "amharic")` -- averaged over all sources within that language
- `("all", "all")` -- averaged over the per-language averages (each language weighted
  equally, regardless of completion count)

In [11]:
outputs = [
    {"source": "okapi", "language": "igbo", "completion": three_completions[0]},
    {"source": "okapi", "language": "igbo", "completion": three_completions[1]},
    {"source": "dolly", "language": "igbo", "completion": three_completions[2]},
    {"source": "okapi", "language": "amharic", "completion": three_completions_amharic[0]},
    {"source": "okapi", "language": "amharic", "completion": three_completions_amharic[1]},
]

all_metrics = cm.compute_all_metrics(outputs)
for key, metrics in all_metrics.items():
    print(key, "->", metrics)

Completion:  Ndị mmadụ nwere ike ịrụ ọrụ ubi dị iche iche iji enwe mmasị, 
 Ọ bụrụ na ị nọ n'ụzọ dị mfe ma ọ bụ na ị nọ n'ụzọ na-enyere aka, ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe. Mgbe ụbọchị dị mfe, ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe
Normalized Completion:  Ndị mmadụ nwere ike ịrụ ọrụ ubi dị iche iche iji enwe mmasị 
 Ọ bụrụ na ị nọ nụzọ dị mfe ma ọ bụ na ị nọ nụzọ naenyere aka ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe Mgbe ụbọchị dị mfe ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe
Lines:  ['Ndị mmadụ nwere ike ịrụ ọrụ ubi dị iche iche iji enwe mmasị ', ' Ọ bụrụ na ị nọ nụzọ dị mfe ma ọ bụ na ị nọ nụzọ naenyere aka ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe Mgbe ụbọchị dị mfe ọ nwere ike ime ka ị rụ ọrụ ubi dị mfe']
line_tokens:  [['Ndị', 'mmadụ', 'nwere', 'ike', 'ịrụ', 'ọrụ', 'ubi', 'dị', 'iche', 'iche', 'iji', 'enwe', 'mmasị'], ['Ọ', 'bụrụ', 'na', 'ị', 'nọ', 'nụzọ', 'dị', 'mfe', 'ma', 'ọ', 'bụ', 'na', 'ị', 'nọ', 'nụzọ', 'naenyere', 'aka', 'ọ', 'nwere', 'ike', 'ime', 'ka', 'ị', 'rụ', 'ọrụ', 